# Strategy 16 — Deep Reinforcement Learning Trading Agent

> **Status: PLAN ONLY — no code cells yet.** This notebook is the human-readable
> review surface for the implementation plan. Each section below describes
> what the eventual code cell will do, what it depends on, and which design
> decisions are still open.
>
> See [[../DocumentationVault/strategies/16_Deep_RL_Trading]] for the strategy
> specification (Obsidian).

---

## §0. Prerequisites and environment

This notebook trains deep RL policies and **requires CUDA**. Before running:

- **GPU:** NVIDIA RTX 3060 Laptop (6 GB) — already present on the dev box.
- **Driver:** ≥ 560 (CUDA 12.6 runtime). `nvidia-smi` should report a CUDA
  version of 12.x or 13.x.
- **Python:** the repo's `.venv` runs Python 3.14.4. PyTorch's stable cu126
  wheels may not yet publish 3.14 builds at the time of writing — see the
  install instructions below for the nightly / Python-3.12 fallback paths.

### Install commands (run **outside** the notebook, in the shell)

```bash
# 1. PyTorch CUDA build — stable wheel from PyTorch's own index
source .venv/bin/activate.fish
pip install --upgrade pip
pip install --index-url https://download.pytorch.org/whl/cu126 torch torchvision

# 2. If Python 3.14 stable wheels aren't published yet, try nightly:
#    pip install --pre --index-url https://download.pytorch.org/whl/nightly/cu126 torch torchvision

# 3. RL stack
pip install "stable-baselines3[extra]>=2.3" "gymnasium>=0.29"

# 4. Verify
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

### Data dependency

The notebook reads multi-timeframe parquet caches built by `source.spark_loader`.
**That module is currently only on the `feature/multi-filter-portfolio-system`
branch** — it must be cherry-picked into this branch (or that PR merged into
`main`) before the data cells run. See the strategy doc's *Implementation Notes*
section for the rationale.

JDK 17 or 21 is required for PySpark 4.x. `get_spark()` auto-detects; if the
only JDK on this box is 26, install one of the supported versions:

```bash
sudo pacman -S jdk21-openjdk    # Arch / CachyOS
```

### Reference links

- PyTorch install selector: <https://pytorch.org/get-started/locally/>
- stable-baselines3 docs: <https://stable-baselines3.readthedocs.io/>
- Gymnasium docs: <https://gymnasium.farama.org/>


## §1. Imports

The eventual code cell will import:

- **Stdlib:** `gc`, `os`, `sys`, `pathlib.Path`, `dataclasses`, `random`.
- **PyData:** `numpy`, `pandas`, `matplotlib.pyplot`.
- **PyTorch:** `torch` — also asserts `torch.cuda.is_available()` and prints
  `torch.cuda.get_device_name(0)`. If CUDA is unavailable the cell prints a
  loud warning and continues with `device="cpu"` (debug only — training will
  be infeasibly slow).
- **RL stack:** `gymnasium as gym`, `stable_baselines3` (`PPO`, `DQN`),
  `stable_baselines3.common.callbacks` (`EvalCallback`, `CheckpointCallback`),
  `stable_baselines3.common.vec_env` (`DummyVecEnv`, `SubprocVecEnv`),
  `stable_baselines3.common.monitor.Monitor`.
- **Repo `source` imports:**
    - `source.spark_loader` — `get_spark`, `build_spark_grid` (parquet
      multi-TF cache; see §0 dependency note).
    - `source.backtest.Backtester`, `source.metrics.compute_metrics`,
      `source.dashboard.plot_backtest_dashboard`.
    - `source.robustness` — `block_bootstrap_trades`, `subperiod_analysis`,
      `parameter_sensitivity`, `monte_carlo_trades`.
    - `source.comparison.STRATEGY_REGISTRY` (read-only, for benchmark
      comparison in §13).
    - `source.parallel.parallel_map` (only used in §13 if running benchmarks
      in parallel).
- **New (to be added under `source/rl/`):**
    - `source.rl.env.TradingEnv`
    - `source.rl.train` — `train_one_seed`, `latest_checkpoint`,
      `evaluate_policy_to_signals`, `make_vec_env`.
- **Random seeds** pinned globally: `random.seed`, `np.random.seed`,
  `torch.manual_seed`, `torch.cuda.manual_seed_all` — all set from
  `GLOBAL_SEED = 42`. SB3 seeds are derived per training run.


## §2. Configuration

Single dict cell pinning every knob so the rest of the notebook reads as
"apply config to data, env, agent, evaluator". Mirrors the layout of
`technical_analysis/15_multi_filter_portfolio_system.ipynb` §2.

### Markets and timeframes

```python
GROUP_TIMEFRAMES = {
    "forex": ["1h", "4h"],          # 1D dropped — too few bars for RL
    "b3":    ["30min", "1h"],
    # "crypto": [...]               # skipped — no data/crypto/
}
ASSETS = {
    "forex": ["EURUSD", "EURCAD", "GBPCHF"],
    "b3":    ["WDO", "WIN"],
}
WFO_ASSETS = {                       # hyperparameter selection happens on these
    "forex": ["EURUSD"],
    "b3":    ["WIN"],
}
```

### Train / validation / OOS split

```python
SPLITS = {
    "forex": {"train": ("2016-01-01", "2022-12-31"),
              "val":   ("2023-01-01", "2023-12-31"),
              "oos":   ("2024-01-01", "2026-12-31")},
    "b3":    {"train": ("2021-01-01", "2023-12-31"),
              "val":   ("2024-01-01", "2024-09-30"),
              "oos":   ("2024-10-01", "2026-12-31")},
}
```

### RL hyperparameters

```python
RL_CONFIG = dict(
    algorithms          = ["PPO", "DQN"],
    n_seeds             = 3,
    total_timesteps     = 1_000_000,
    n_envs              = 8,
    eval_freq           = 50_000,
    save_freq           = 100_000,
    obs_window          = 32,
    episode_len         = 2048,
    gamma               = 0.99,
    tx_cost_bps_by_grp  = {"forex": 5, "b3": 10},
    policy_arch         = (64, 64),
    device              = "cuda",
)
```

### Disabled-feature flags (per repo convention)

```python
DISABLED_V1 = dict(
    use_continuous_action       = False,
    use_differential_sharpe     = False,
    use_drawdown_penalty        = False,
    use_holding_penalty         = False,
    use_lstm_policy             = False,
    use_cnn_policy              = False,
    use_flat_close_signal       = False,
)
```

Open question for review: should `WFO_ASSETS` ever include a *second*
representative asset per group to guard against single-asset hyperparameter
overfit? The trade-off is roughly 2× training cost.


## §3. Data — PySpark multi-timeframe load

Reuses the `source.spark_loader` module from the multi-filter system. The
goal here is the same as in notebook 15: never hold every CSV in RAM, fan out
worker processes that each open a small parquet slice.

The cell will:

1. `get_spark(java_home=...)` — instantiates a Spark session pinned to a
   compatible JDK (17 or 21). Errors loudly if only JDK 24+ is found.
2. `build_spark_grid(...)` — for every M1 source CSV, tumbles into the target
   timeframes (`1h`, `4h`, `30min`) and writes parquet keyed by source mtime.
   This is a one-time cost per data update.
3. `read_parquet_slice(group, tf, asset, start, end)` — returns a small
   `pd.DataFrame` for the requested chronological slice (train / val / oos).
   Used by the env and by the evaluation pipeline.
4. Sanity prints: per `(group, tf, asset)` cell, bar count and date range for
   each split.

Per the parallelism convention (see CLAUDE memory): the parent process never
holds more than one DataFrame at a time — each training worker opens its own
parquet slice inside the worker process.

**Dependency reminder:** `source/spark_loader.py` is not on `main` yet —
either cherry-pick from `feature/multi-filter-portfolio-system` or wait for
that PR to merge. See §0.


## §4. Data cleaning and preprocessing

For each `(group, tf, asset)` slice loaded by §3:

- **Drop rows with NaN OHLCV** (rare for the source data, but defensive).
- **Assert monotonic index** — `df.index.is_monotonic_increasing`. Raise if
  not (the Spark loader should already guarantee this).
- **Drop session-boundary partials** (B3 only) — first/last bar of each session
  often has stub volume. Drop where `tick_vol == 0` or `volume_ratio < 0.05`.
- **Chronological train/val/OOS split** — slice the frame into three
  non-overlapping windows by date. Print bar counts per split per cell.
- **Online normalisation only** — no global `StandardScaler.fit(train_df)`
  that leaks distribution-level info into the env. All z-scoring happens
  inside `TradingEnv` against a rolling lookback (default 252 bars).

No global feature table is materialised — that would defeat the lazy/parallel
design. Each env recomputes features from its own slice inside the worker
process.


## §5. Feature engineering and technical indicators

The set of features the env exposes to the policy (also defined in the
strategy doc):

| Feature | Formula | Notes |
|---|---|---|
| `log_return` | `log(close_t / close_{t-1})` | 1-bar log return |
| `realized_vol` | rolling std of `log_return` over `rv_window=20` | |
| `rsi / 100` | Wilder RSI period 14, scaled to `[0, 1]` | |
| `atr_rel` | ATR-14 / close | unit-free volatility |
| `volume_ratio` | `tick_vol / SMA(tick_vol, vol_period=20)` | |
| `bb_pos` | `(close − bb_mid) / (bb_upper − bb_mid)` | ∈ `[−1, +1]` band position |
| `current_position` | `{−1, 0, +1}` carried from previous step's action | |
| `bars_in_position` | step count since last position change, normalised by `episode_len` | |

Implementation:

- All windowed features are computed via pandas `rolling(...)` once per
  episode (or once per env instantiation, then sliced per step) — *not*
  per-step, which would be O(n²).
- After the rolling features are computed, the env z-scores each column
  against a rolling `rolling_z_window=252`-bar lookback. The z-score uses
  *only* bars up to and including `t` (no look-ahead). The first
  `rolling_z_window` bars of each episode are skipped on `reset()`.
- The observation passed to the policy at step `t` is a flattened
  `obs_window × n_features` window plus the 2 position-state scalars.

**Open question for review:** is `obs_window = 32` too long (more parameters,
slower training) or too short (policy can't see weekly seasonality)?
Candidate WFO range `{16, 32, 64}`.


## §6. Environment — `TradingEnv(gymnasium.Env)`

New module: `source/rl/env.py`. The env is the core RL abstraction — all the
trading semantics live here, not in the agent. Code-cell-to-be:

```python
class TradingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, df, params: DeepRLTradingParams, mode: str = "train"):
        # mode ∈ {"train", "eval"}; train picks random slices, eval is deterministic full pass
        ...

    def reset(self, seed=None, options=None):
        # 1. Pick episode start (random for train, 0 for eval).
        # 2. Reset position, equity, bars_in_position.
        # 3. Warm-up rolling stats over first rolling_z_window bars.
        # 4. Return (obs, info).
        ...

    def step(self, action):
        # 1. Decode action ∈ {0, 1, 2} → target_position ∈ {−1, 0, +1}.
        # 2. Apply B3 session mask (force flat outside session).
        # 3. Compute reward = position_{t-1} · log_return_t − tx_cost · |Δposition|.
        # 4. Update position, advance index, recompute observation.
        # 5. Terminate on end-of-window or drawdown > max_drawdown_fraction.
        # 6. Return (obs, reward, terminated, truncated, info).
        ...

    @property
    def observation_space(self):
        # Box(shape=(obs_window * n_features + 2,))
        ...

    @property
    def action_space(self):
        # Discrete(3)
        ...
```

Subtleties to document in the code:

- **Reward timing.** The reward at bar `t` is earned by the position held
  *into* bar `t` (i.e. set by the action at `t-1`). The very first action of
  an episode contributes zero reward at step 0; reward begins at step 1.
- **Transaction cost** is charged on the bar where the action differs from
  the previous position, not on entry / exit specifically. This is a more
  honest representation for a continuous-action policy and a non-issue for
  discrete.
- **Drawdown circuit-breaker** terminates the episode (not a real-money
  stop-loss) — it prevents the policy from learning that "ride it out
  forever" is a valid strategy.
- **Vectorisation.** SB3 wraps with `Monitor` + `DummyVecEnv` / `SubprocVecEnv`.
  Use `SubprocVecEnv(n_envs=8)` for training (multi-process); `DummyVecEnv`
  for evaluation (single-process determinism).

**Open question for review:** termination on drawdown > 30 % — what fraction
of training episodes are expected to be terminated by this? If it's near
zero the circuit breaker is harmless; if it's high the policy may learn an
over-cautious profile. Worth a sanity print in the eventual code cell.


## §7. Model definition — PPO and DQN

Both algorithms instantiated identically except for class:

```python
def make_model(algo: str, env, params: DeepRLTradingParams, log_dir: Path):
    common = dict(
        policy        = "MlpPolicy",
        env           = env,
        verbose       = 0,
        device        = params.device,
        tensorboard_log = str(log_dir),
        policy_kwargs = dict(net_arch=list(params.policy_arch)),
        seed          = params.training_seed,
    )
    if algo == "PPO":
        return PPO(**common,
                   learning_rate = 3e-4,
                   n_steps       = 2048,
                   batch_size    = 64,
                   n_epochs      = 10,
                   gamma         = params.gamma,
                   gae_lambda    = 0.95,
                   clip_range    = 0.2,
                   ent_coef      = 0.01,
                   vf_coef       = 0.5)
    elif algo == "DQN":
        return DQN(**common,
                   learning_rate           = 1e-4,
                   buffer_size             = 100_000,
                   learning_starts         = 10_000,
                   batch_size              = 64,
                   tau                     = 1.0,
                   gamma                   = params.gamma,
                   train_freq              = 4,
                   target_update_interval  = 1000,
                   exploration_fraction    = 0.1,
                   exploration_final_eps   = 0.05)
    raise ValueError(algo)
```

Architecture choices and their justifications are tabulated in the strategy
doc — this section's markdown summary will link to it rather than duplicate.


## §8. Resume from previous training (checkpoint discovery)

This section is the core of "training is interruptible". The code cell will:

1. **Discover checkpoints** under `models/16_deep_rl/`.
2. For each `(algo, group, tf, asset, seed)` cell, find the latest
   `step<N>.zip` and (for DQN) its replay buffer pickle.
3. Build a tidy DataFrame:

   ```
   algo  group  tf     asset    seed  step_done  has_best
   PPO   forex  4h     EURUSD   0     400_000    True
   PPO   forex  4h     EURUSD   1     —          False
   ...
   ```

4. **Print a resume plan** — for each cell, either "resume from step N" or
   "start from scratch". The user reads this *before* §9 kicks off training
   to confirm nothing unexpected is being overwritten.

Helper to be added in `source/rl/train.py`:

```python
def latest_checkpoint(model_dir: Path, algo: str, group: str, tf: str,
                       asset: str, seed: int) -> tuple[Path | None, int]:
    ...  # returns (path_or_None, steps_already_trained)
```

**Why this matters:** RL training is the most fragile / expensive step in
the notebook. Treating checkpoints as first-class (and human-visible in §8
before training starts) is the cheapest insurance against "I trained for
2 hours and lost it".


## §9. Model training

Per-cell training loop. The outer iteration is over the
`{PPO, DQN} × WFO_ASSETS × n_seeds` grid for hyperparameter selection
(§9.1), then over the full `ASSETS × n_seeds` grid for the chosen
hyperparameters (§9.2). Training is fanned out across processes via
`parallel_map` only at the *seed* level — multiple algorithms or assets
training in parallel would oversubscribe the GPU.

### 9.1 Hyperparameter selection on `WFO_ASSETS`

For each cell in `{PPO, DQN} × WFO_ASSETS`:

- Build a small grid: `gamma × learning_rate × policy_arch`
  (3 × 3 × 3 = 27 combos).
- Train `n_seeds = 3` per combo (81 training runs per cell).
- For each run:
    - Construct `TradingEnv` on the **train** split.
    - Construct a separate `TradingEnv` on the **val** split for
      `EvalCallback`.
    - Train for `total_timesteps`, saving via `CheckpointCallback` every
      `save_freq` steps and the best-by-val-reward via `EvalCallback`.
    - Save final + best to `models/16_deep_rl/...`.
- Rank by mean validation Sharpe across seeds; pick the best combo per
  `(algo, group)` cell. Store in `BEST_HP[(algo, group)]`.

### 9.2 Generalisation pass — full asset grid

For each cell in `{PPO, DQN} × ASSETS \ WFO_ASSETS`:

- Use `BEST_HP[(algo, group)]` from §9.1.
- Train `n_seeds = 3` runs on each asset's train split.
- Save best-by-val and final checkpoints.

### Training observability

- `TensorBoard` logs to `models/16_deep_rl/tb/` (gitignored).
- A summary cell at the bottom of §9 prints, per cell:
  best validation reward, training time, GPU memory peak.

**Open question for review:** the §9.1 grid is 81 runs × ~30 min = ~40 GPU
hours for forex alone. Is the time budget acceptable? Alternatives:
- Use Optuna for ~20 trials instead of grid (probably better).
- Reduce `total_timesteps` to 500k for the grid pass, 1M only for the
  generalisation pass.
- Drop `policy_arch` from the grid (saves 3×).


## §10. WFO of the trained policy — temporal generalisation only

Re-uses `source.wfo.walk_forward` in a non-standard way: instead of WFO-ing
*strategy hyperparameters*, we WFO-evaluate the **same trained policy**
across consecutive OOS chunks. The grid has a single combo (the trained
policy), and the folds carve the OOS window into 3 chronological slices.

This answers a specific question: *does the policy's edge decay over the
OOS window, or is it stationary?* A canonical hyperparameter WFO is
intentionally out of scope for the RL training itself (see §9.1 reasoning).

The cell will:

1. For each `(algo, group, tf, asset)` cell with a best-checkpoint from §9.
2. Load the best-by-validation checkpoint.
3. `evaluate_policy_to_signals(model, oos_df, deterministic=True)` →
   pre-computed signal array.
4. Wrap with `DeepRLTradingStrategy` (signal-replay) and feed to
   `Backtester.run(...)`.
5. Run `walk_forward(...)` with `n_folds=3` over OOS to get fold-level
   metrics.
6. Plot `plot_wfo_dashboard(...)` per cell — equity curves and metric
   evolution across folds.


## §11. Full backtest — every `(group, tf, asset)` cell

For each `(algo, group, tf, asset)` cell:

1. Pick the best checkpoint across seeds (max mean-validation-reward).
2. `evaluate_policy_to_signals(model, full_df, deterministic=True)` — *full
   df* meaning train + val + oos concatenated, so the per-fold equity curve
   is comparable to the other strategies' baseline backtests.
3. Wrap with `DeepRLTradingStrategy(signal_array=..., params=...)` and run
   through `Backtester`.
4. Plot `plot_backtest_dashboard(...)` per cell — equity curve, trades,
   drawdowns, metrics panel.
5. Build a single tidy `metrics_df` keyed by
   `(algo, group, tf, asset, seed)`. Pivot to a wide table for inclusion
   in the strategy-comparison dashboard (§13).

Note on cost double-counting: the env already applies `tx_cost_bps` inside
the reward, but the `Backtester` does **not** apply commissions/slippage by
default (`slippage_points=0.0`). They are not double-counted because the
Backtester uses the policy's signal-replay path, not the env's reward path.
This is called out in the §11 markdown summary.


## §12. Overfitting and robustness checks

The classical robustness suite from [[06_Robustness_Testing]], adapted to RL:

### 12.1 Seed sensitivity — RL-specific

The single most important RL robustness check. For each
`(algo, group, tf, asset)` cell:

- Plot OOS equity curves of all `n_seeds` seeds on the same axes.
- Report **mean ± std** of OOS Sharpe / Profit Factor across seeds.
- If std/mean > 1 (high relative variance), flag the cell as "unstable —
  the policy is at the mercy of seed luck".

### 12.2 Block bootstrap on OOS trades

`block_bootstrap_trades(trades_df, block_size_bars=...)` per cell. Reports
the bootstrap distribution of OOS Sharpe and `P(Sharpe > 0)`.

### 12.3 Sub-period analysis

`subperiod_analysis(trades_df, freq="YE")` per cell. Confirms the policy
isn't carried by a single year of OOS performance.

### 12.4 Parameter sensitivity — *post-training*

`parameter_sensitivity(...)` over a small grid of *inference-time* knobs:

- `deterministic ∈ {True, False}` — does stochastic policy roll-out
  meaningfully change outcomes?
- `tx_cost_bps × {0.5, 1.0, 2.0}` — replay through the Backtester with a
  cost overlay (the eval-time cost can differ from the train-time cost) to
  measure cost sensitivity.

(Note: classical hyperparameter sensitivity requires re-training the policy
per point — already covered by §9.1's grid.)

### 12.5 Synthetic asset null hypothesis

Same H₀-overfitted-hypothesis-test framing as notebook 15 §7: train the
policy on a Geometric-Brownian-Motion synthetic asset and verify that OOS
performance drops to ~0 (no real edge to learn from). If the policy still
"works" on synthetic data, it's overfitting to noise.

This is the most diagnostic single check for an RL policy. Done as a single
quick training run per algo on a synthetic of matched volatility, **not** as
a full sweep.


## §13. Comparison with benchmarks

Side-by-side OOS performance of the RL policy vs the established repo
strategies, evaluated on the **same** OOS window and the **same** asset cells.

### Benchmarks pulled from the existing repo

- **Strategy 01 — SMA Crossover ATR Risk.** The unoptimised baseline; the
  bar an RL policy must clear to be interesting at all.
- **Strategy 10 — HMM Regime Filter (GaussianMixture proxy).** The repo's
  other ML-based strategy. Comparing against #10 separates the "learned vs
  hand-coded" axis (RL vs SMA) from the "ML vs ML" axis (RL vs HMM).
- **Buy-and-hold.** Trivial but essential — if RL doesn't beat passive on
  the OOS window, no further analysis is needed.

### Mechanics

- Use `source.comparison.STRATEGY_REGISTRY` to pull the baseline configs.
- Run #01 and #10 through `Backtester` on the same OOS slices the RL
  policies are evaluated on (cache lookup if the `comparison/.cache/` hit
  is fresh enough; otherwise rerun).
- Build a single comparison table (rows = strategies, columns = OOS metrics
  per cell).
- Plot `plot_strategy_equity_overlay(...)` for the top-N cells.

### Acceptance criteria (pre-defined, copied from strategy doc)

The RL agent **passes** the comparison only if:

- **Mean OOS Sharpe across seeds beats both SMA #01 and HMM #10** on at
  least 2 of the 4 representative cells (forex-1h-EURUSD, forex-4h-EURUSD,
  b3-30min-WIN, b3-1h-WIN), **and**
- **Worst-seed Sharpe is positive** on those cells.

Otherwise: report negative result in §14 and leave Status = "Backtested —
did not beat baseline".


## §14. Findings and next steps

Populated post-run. The cell will summarise:

- Whether the acceptance criteria from §13 were met.
- Which algorithm/cell combinations were stable across seeds vs which were
  seed-luck.
- The §12.5 synthetic-null result (the single most credible robustness
  signal).
- A ranked list of v2 follow-ups, draft of which already lives in the
  strategy doc's *Known Weaknesses* section:
    - Enable `use_differential_sharpe` and A/B against per-step PnL reward.
    - Multi-asset env (cycle assets per episode).
    - HMM-state pre-conditioning (ensemble #10 + #16).
    - LSTM policy via `sb3_contrib.RecurrentPPO`.
    - "Policy mode" Backtester extension (`use_flat_close_signal`).
    - Continuous position sizing (`use_continuous_action`) — requires
      `PortfolioBacktester` to be on `main`.
- Open issue numbers (TBD on PR open) for each follow-up.

Notebook commits **unexecuted**, per repo convention.
